In [1]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [2]:
print(train.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [3]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train['label'] = train["answer"].map(label_map)
train.head()

,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,0


## Q1

In [4]:
idx = 150

print("Original answer:", train.loc[idx, "answer"])
print("Encoded label :", train.loc[idx, "label"])

Original answer: C
Encoded label : 2


## Q2

In [5]:
idx = 0

formatted_input = (
    str(train.loc[idx, "prompt"]) +
    " [SEP] " +
    str(train.loc[idx, "B"])
)

print(formatted_input)

print("\nCharacter Length:", len(formatted_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.

Character Length: 407


## Q3

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

idx = 0

choices=[]

for option in ["A","B","C","D","E"]:
    text = (str(train.loc[idx,"prompt"])+" [SEP] "+str(train.loc[idx, option]))
    choices.append(text)
choices

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

["Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.",
 "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.",
 "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among th

In [7]:
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

print("input_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)
print("Second dimension:", input_ids.shape[1])

input_ids shape: torch.Size([1, 5, 128])
attention_mask shape: torch.Size([1, 5, 128])
Second dimension: 5


## Q4

In [8]:
import torch
batch_input_ids = []
batch_attention_mask = []

for idx in range(16):
    choices = [
        f"{train.loc[idx, 'prompt']} [SEP] {train.loc[idx, option]}"
        for option in ["A", "B", "C", "D", "E"]
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    batch_input_ids.append(encoding["input_ids"])
    batch_attention_mask.append(encoding["attention_mask"])

input_ids = torch.stack(batch_input_ids)
attention_mask = torch.stack(batch_attention_mask)

print("input_ids shape:", input_ids.shape)

total_positions = input_ids.numel()
print("Total token positions:", total_positions)

input_ids shape: torch.Size([16, 5, 128])
Total token positions: 10240


## Q5

In [9]:
from transformers import AutoModelForMultipleChoice

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

idx = 0

choices = [
    f"{train.loc[idx, 'prompt']} [SEP] {train.loc[idx, option]}"
    for option in ["A", "B", "C", "D", "E"]
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0)
}

if "token_type_ids" in encoding:
    inputs["token_type_ids"] = encoding["token_type_ids"].unsqueeze(0)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print("Logits shape:", logits.shape)
print("Number of logits for one question:", logits.shape[1])
print("Logits:", logits)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Number of logits for one question: 5
Logits: tensor([[0.1010, 0.1196, 0.1143, 0.0800, 0.0947]])


## Q6

In [10]:
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

idx = 0

choices = [
    f"{train.loc[idx, 'prompt']} [SEP] {train.loc[idx, option]}"
    for option in ["A", "B", "C", "D", "E"]
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0),
}

if "token_type_ids" in encoding:
    inputs["token_type_ids"] = encoding["token_type_ids"].unsqueeze(0)

label = torch.tensor([label_map[train.loc[idx, "answer"]]])

with torch.no_grad():
    outputs = model(**inputs, labels=label)

loss = outputs.loss

print("Loss:", loss)
print("Loss shape:", loss.shape)
print("Number of dimensions:", loss.ndim)

Loss: tensor(1.5919)
Loss shape: torch.Size([])
Number of dimensions: 0


## Q7

In [11]:
# !pip install -q peft

from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Percentage trainable: {100 * trainable_params / total_params:.4f}%")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 295,681
Total parameters: 109,778,690
Percentage trainable: 0.2693%


## Q8

In [12]:
# !pip install -q datasets

from transformers import AutoTokenizer
from datasets import Dataset

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Encode labels
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

examples = []

# Use first 100 rows
for idx in range(100):

    # Create the 5 formatted inputs
    choices = [
        f"{train.loc[idx, 'prompt']} [SEP] {train.loc[idx, option]}"
        for option in ["A", "B", "C", "D", "E"]
    ]

    # Tokenize
    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    examples.append({
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": label_map[train.loc[idx, "answer"]]
    })

# Create Hugging Face Dataset
dataset = Dataset.from_list(examples)

print(dataset)

# Inspect first example
print("input_ids shape:",
      len(dataset[0]["input_ids"]),
      "x",
      len(dataset[0]["input_ids"][0]))

print("Number of tokenized choices:",
      len(dataset[0]["input_ids"]))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})
input_ids shape: 5 x 128
Number of tokenized choices: 5


## Q9

In [13]:
# !pip install -q transformers datasets peft accelerate

import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

# -----------------------------
# Load tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

# -----------------------------
# Prepare first 32 examples
# -----------------------------
examples = []

for idx in range(32):

    choices = [
        f"{train.loc[idx,'prompt']} [SEP] {train.loc[idx,opt]}"
        for opt in ["A","B","C","D","E"]
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    examples.append({
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": label_map[train.loc[idx,"answer"]]
    })

dataset = Dataset.from_list(examples)

# -----------------------------
# Load model
# -----------------------------
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# -----------------------------
# Apply LoRA
# -----------------------------
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, config)

# -----------------------------
# Training arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir="./outputs",
    max_steps=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

# -----------------------------
# Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

print("\nFinal global_step:", trainer.state.global_step)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,3.114570
2,3.081212
3,3.186552
4,3.310529



Final global_step: 4


## Q10

In [14]:
import torch
model = model.cpu()
# Put model in evaluation mode
model.eval()

idx = 0

# Create the 5 formatted inputs
choices = [
    f"{train.loc[idx, 'prompt']} [SEP] {train.loc[idx, opt]}"
    for opt in ["A", "B", "C", "D", "E"]
]

# Tokenize
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,   # Same as used during training
    return_tensors="pt"
)

# Prepare inputs
inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0),
}

if "token_type_ids" in encoding:
    inputs["token_type_ids"] = encoding["token_type_ids"].unsqueeze(0)

# Inference
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

# Convert logits to probabilities
probs = torch.softmax(logits, dim=1)

print("Logits:")
print(logits)

print("\nProbabilities:")
print(probs)

# Probability of Option E
option_e_prob = probs[0, 4].item()

print("\nProbability of Option E:", round(option_e_prob, 4))

Logits:
tensor([[0.2522, 0.2227, 0.2290, 0.2075, 0.2185]])

Probabilities:
tensor([[0.2053, 0.1993, 0.2006, 0.1963, 0.1985]])

Probability of Option E: 0.1985
